# 🚀 Stacked Ensemble: Ekman 7 + 2-Level Class Weighting (Full Dataset 43k)

**สถาปัตยกรรม:** WangchanBERTa + mBERT + XLM-RoBERTa -> Level-2 Weighted Random Forest Meta-Classifier  
**ชุดข้อมูล:** GoEmotions Full Dataset (Train 43,410 แถว, Val 5,426 แถว, Test 5,131 แถว)  
**Taxonomy:** Ekman 7 กลุ่มอารมณ์ (`Joy`, `Anger`, `Sadness`, `Surprise`, `Fear`, `Disgust`, `Neutral`)  
**เทคนิคเสริม:**
1. **FP16 Mixed Precision:** เทรนไวขึ้น 2-3 เท่าด้วย Tensor Cores
2. **Dynamic VRAM Management:** สลับโมเดลเข้า-ออกจาก GPU ทีละตัว VRAM ไม่ล้น 8 GB แน่นอน
3. **2-Level Class Weighting:**
   - Level 1: `pos_weight` ใน `BCEWithLogitsLoss` พร้อม Boost Factor (`Fear x3.0`, `Disgust x3.0`, `Sadness x2.5`)
   - Level 2: `class_weight` แยกอิสระในแต่ละ Estimator ของ Random Forest

## Step 1: Import ไลบรารี & กำหนดค่า Device (GPU/CPU)

In [3]:
import os
import ast
import json
import joblib
import gc
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from torch.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, hamming_loss, f1_score
from tqdm.auto import tqdm

# เชื่อมต่อ eval_logger ใน src/
if '..' not in sys.path and '.' not in sys.path:
    sys.path.append('..')
elif '.' in sys.path:
    sys.path.append('.')
from src.eval_logger import save_eval_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Running on device: {device}')

SAVE_DIR = 'saved_models_ekman_weighted'
if not os.path.exists(SAVE_DIR) and os.path.exists('../saved_models'):
    SAVE_DIR = '../saved_models_ekman_weighted'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'📁 Save directory: {SAVE_DIR}')

🚀 Running on device: cuda
📁 Save directory: ../saved_models_ekman_weighted


## Step 2: โหลดชุดข้อมูลเต็ม 43k & แมปปิ้ง Ekman 7 คลาส

In [4]:
TRAIN_PATH = '../goemotions/data/go_emotions_train_with_translation.csv'
VAL_PATH   = '../goemotions/data/go_emotions_validation_with_translation.csv'
TEST_PATH  = '../goemotions/data/go_emotions_test_with_translation.csv'

if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'goemotions/data/go_emotions_train_with_translation.csv'
    VAL_PATH   = 'goemotions/data/go_emotions_validation_with_translation.csv'
    TEST_PATH  = 'goemotions/data/go_emotions_test_with_translation.csv'

df_train = pd.read_csv(TRAIN_PATH).dropna(subset=['text_th']).reset_index(drop=True)
df_val   = pd.read_csv(VAL_PATH).dropna(subset=['text_th']).reset_index(drop=True)
df_test  = pd.read_csv(TEST_PATH).dropna(subset=['text_th']).reset_index(drop=True)

print(f'จำนวนข้อมูล Train ภาษาไทย      : {len(df_train):,} แถว')
print(f'จำนวนข้อมูล Validation ภาษาไทย : {len(df_val):,} แถว')
print(f'จำนวนข้อมูล Test ภาษาไทย       : {len(df_test):,} แถว')

ORIGINAL_LABELS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval',
    'caring', 'confusion', 'curiosity', 'desire', 'disappointment',
    'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
    'gratitude', 'grief', 'joy', 'love', 'nervousness',
    'optimism', 'pride', 'realization', 'relief', 'remorse',
    'sadness', 'surprise', 'neutral'
]

EKMAN_MAPPING = {
    'Joy': [
        'admiration', 'amusement', 'approval', 'caring', 'desire',
        'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief'
    ],
    'Anger': ['anger', 'annoyance', 'disapproval'],
    'Sadness': ['disappointment', 'embarrassment', 'grief', 'remorse', 'sadness'],
    'Surprise': ['confusion', 'curiosity', 'realization', 'surprise'],
    'Fear': ['fear', 'nervousness'],
    'Disgust': ['disgust'],
    'Neutral': ['neutral']
}

EKMAN_LABELS = list(EKMAN_MAPPING.keys())
NUM_LABELS = len(EKMAN_LABELS)

ekman_label_to_id = {lbl: i for i, lbl in enumerate(EKMAN_LABELS)}
orig_id_to_ekman_id = {}
for orig_id, orig_lbl in enumerate(ORIGINAL_LABELS):
    for ekman_lbl, orig_list in EKMAN_MAPPING.items():
        if orig_lbl in orig_list:
            orig_id_to_ekman_id[orig_id] = ekman_label_to_id[ekman_lbl]
            break

def map_labels(labels_raw):
    mapped = set()
    for lbl in labels_raw:
        if isinstance(lbl, int) and lbl in orig_id_to_ekman_id:
            mapped.add(orig_id_to_ekman_id[lbl])
        elif isinstance(lbl, str):
            for ek_lbl, orig_list in EKMAN_MAPPING.items():
                if lbl in orig_list:
                    mapped.add(ekman_label_to_id[ek_lbl])
    return list(mapped)

for df in [df_train, df_val, df_test]:
    df['labels_list'] = df['labels'].apply(ast.literal_eval)
    df['ekman_labels'] = df['labels_list'].apply(map_labels)

mlb = MultiLabelBinarizer(classes=list(range(NUM_LABELS)))
Y_train = mlb.fit_transform(df_train['ekman_labels'])
X_train = df_train['text_th'].astype(str).tolist()

Y_val = mlb.transform(df_val['ekman_labels'])
X_val = df_val['text_th'].astype(str).tolist()

Y_test = mlb.transform(df_test['ekman_labels'])
X_test = df_test['text_th'].astype(str).tolist()

print(f'✅ แมปปิ้งเสร็จสมบูรณ์! Train Shape: {Y_train.shape} | Test Shape: {Y_test.shape}')

จำนวนข้อมูล Train ภาษาไทย      : 43,410 แถว
จำนวนข้อมูล Validation ภาษาไทย : 5,426 แถว
จำนวนข้อมูล Test ภาษาไทย       : 5,131 แถว
✅ แมปปิ้งเสร็จสมบูรณ์! Train Shape: (43410, 7) | Test Shape: (5131, 7)


## Step 3: Level 1 Class Weighting (Boosted pos_weight ใน Loss Function)

In [6]:
pos_counts = Y_train.sum(axis=0)
neg_counts = len(Y_train) - pos_counts
base_weight = neg_counts / (pos_counts + 1e-6)

# Boost Factor สำหรับ Minority Classes:
# Joy=0, Anger=1, Sadness=2, Surprise=3, Fear=4, Disgust=5, Neutral=6
BOOST_FACTOR = np.array([
    1.0,   # Joy
    1.0,   # Anger
    2.5,   # Sadness  (boost 2.5x)
    1.0,   # Surprise
    3.0,   # Fear     (boost 3.0x)
    3.0,   # Disgust  (boost 3.0x)
    1.0,   # Neutral
])

boosted_weight = base_weight * BOOST_FACTOR
pos_weight_tensor = torch.FloatTensor(boosted_weight).to(device)

print('⚖️ Boosted pos_weight (ที่จะส่งเข้า BCEWithLogitsLoss):')
for i, lbl in enumerate(EKMAN_LABELS):
    boost_tag = f'>> BOOSTED x{BOOST_FACTOR[i]}' if BOOST_FACTOR[i] > 1.0 else ''
    print(f'  [{i}] {lbl:<10} | pos={int(pos_counts[i]):>5} | pos_weight={boosted_weight[i]:.2f} {boost_tag}')

⚖️ Boosted pos_weight (ที่จะส่งเข้า BCEWithLogitsLoss):
  [0] Joy        | pos=17410 | pos_weight=1.49 
  [1] Anger      | pos= 5579 | pos_weight=6.78 
  [2] Sadness    | pos= 3263 | pos_weight=30.76 >> BOOSTED x2.5
  [3] Surprise   | pos= 5367 | pos_weight=7.09 
  [4] Fear       | pos=  726 | pos_weight=176.38 >> BOOSTED x3.0
  [5] Disgust    | pos=  793 | pos_weight=161.22 >> BOOSTED x3.0
  [6] Neutral    | pos=14219 | pos_weight=2.05 


## Step 4: PyTorch Dataset & ฟังก์ชัน Fine-tuning ด้วย FP16

In [7]:
class ThaiEmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.FloatTensor(self.labels[idx])
        }

def train_model_fp16(model_name, use_fast=True, epochs=3, lr=2e-5, batch_size=16):
    print(f'\n🤖 --- Fine-tuning Base Model: {model_name} (FP16 Accelerated) ---')
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=use_fast)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        problem_type='multi_label_classification'
    ).to(device)

    train_loader = DataLoader(ThaiEmotionDataset(X_train, Y_train, tok), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(ThaiEmotionDataset(X_val,   Y_val,   tok), batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for b in pbar:
            optimizer.zero_grad()
            input_ids = b['input_ids'].to(device)
            mask = b['attention_mask'].to(device)
            labels = b['labels'].to(device)

            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                out = model(input_ids, attention_mask=mask)
                loss = criterion(out.logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]'):
                input_ids = b['input_ids'].to(device)
                mask = b['attention_mask'].to(device)
                labels = b['labels'].to(device)
                with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                    out = model(input_ids, attention_mask=mask)
                    val_loss += criterion(out.logits, labels).item()

        print(f'  ✨ Epoch {epoch+1} -> Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}')

    del optimizer, train_loader, val_loader, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return model, tok

## Step 5: เทรน 3 Base Models ทีละตัว & บันทึกลงไดรฟ์ (Zero VRAM Overflow)

In [5]:
base_configs = [
    ('wangchan', 'airesearch/wangchanberta-base-att-spm-uncased', False),
    ('mbert', 'bert-base-multilingual-cased', True),
    ('xlm_roberta', 'xlm-roberta-base', True)
]

trained_models = {}
trained_toks = {}

for key, pretrained_name, use_fast in base_configs:
    m, tok = train_model_fp16(pretrained_name, use_fast=use_fast, epochs=3, batch_size=16)
    
    # เซฟโมเดลลงไดรฟ์ทันทีที่เทรนเสร็จ
    m.save_pretrained(f'{SAVE_DIR}/{key}_model')
    tok.save_pretrained(f'{SAVE_DIR}/{key}_tokenizer')
    print(f'💾 บันทึกโมเดล {key} สำเร็จ!')
    
    # ดีดโมเดลกลับไปพักไว้ที่ CPU เพื่อคืน VRAM ให้โมเดลตัวถัดไป
    m.to('cpu')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    
    trained_models[key] = m
    trained_toks[key] = tok

print('\n🎉 เทรนและบันทึก Base Models ครบทั้ง 3 ตัวเรียบร้อย!')


🤖 --- Fine-tuning Base Model: airesearch/wangchanberta-base-att-spm-uncased (FP16 Accelerated) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

Epoch 1/3 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

  ✨ Epoch 1 -> Train Loss: 1.4978 | Val Loss: 1.3585


... (30 outputs left out)

Epoch 3/3 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]

  ✨ Epoch 3 -> Train Loss: 1.5135 | Val Loss: 1.5223


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 บันทึกโมเดล xlm_roberta สำเร็จ!

🎉 เทรนและบันทึก Base Models ครบทั้ง 3 ตัวเรียบร้อย!


## Step 6: สกัด 21-dim Probability Features (7 + 7 + 7) อย่างปลอดภัย

In [6]:
def extract_probs_safe(model, tok, texts, name, batch_size=32):
    print(f'\n🔮 กำลังสกัด Feature จาก {name}...')
    model.to(device)
    model.eval()
    ds = ThaiEmotionDataset(texts, np.zeros((len(texts), NUM_LABELS)), tok)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    probs = []
    with torch.no_grad():
        for b in tqdm(loader, desc=f'Extracting {name}'):
            input_ids = b['input_ids'].to(device)
            mask = b['attention_mask'].to(device)
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                out = model(input_ids, attention_mask=mask)
                probs.append(torch.sigmoid(out.logits).cpu().numpy())
    model.to('cpu')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return np.concatenate(probs, axis=0)

train_w = extract_probs_safe(trained_models['wangchan'], trained_toks['wangchan'], X_train, 'WangchanBERTa (Train)')
test_w  = extract_probs_safe(trained_models['wangchan'], trained_toks['wangchan'], X_test,  'WangchanBERTa (Test)')

train_m = extract_probs_safe(trained_models['mbert'], trained_toks['mbert'], X_train, 'mBERT (Train)')
test_m  = extract_probs_safe(trained_models['mbert'], trained_toks['mbert'], X_test,  'mBERT (Test)')

train_x = extract_probs_safe(trained_models['xlm_roberta'], trained_toks['xlm_roberta'], X_train, 'XLM-RoBERTa (Train)')
test_x  = extract_probs_safe(trained_models['xlm_roberta'], trained_toks['xlm_roberta'], X_test,  'XLM-RoBERTa (Test)')

# รวม Feature 21 มิติ
X_train_meta = np.concatenate([train_w, train_m, train_x], axis=1)
X_test_meta  = np.concatenate([test_w, test_m, test_x], axis=1)

print(f'\n✅ รวบรวม Probability สำเร็จ! Shape: Train = {X_train_meta.shape} | Test = {X_test_meta.shape}')


🔮 กำลังสกัด Feature จาก WangchanBERTa (Train)...


Extracting WangchanBERTa (Train):   0%|          | 0/1357 [00:00<?, ?it/s]


🔮 กำลังสกัด Feature จาก WangchanBERTa (Test)...


Extracting WangchanBERTa (Test):   0%|          | 0/161 [00:00<?, ?it/s]


🔮 กำลังสกัด Feature จาก mBERT (Train)...


Extracting mBERT (Train):   0%|          | 0/1357 [00:00<?, ?it/s]

... (3 outputs left out)

Extracting XLM-RoBERTa (Train):   0%|          | 0/1357 [00:00<?, ?it/s]


🔮 กำลังสกัด Feature จาก XLM-RoBERTa (Test)...


Extracting XLM-RoBERTa (Test):   0%|          | 0/161 [00:00<?, ?it/s]


✅ รวบรวม Probability สำเร็จ! Shape: Train = (43410, 21) | Test = (5131, 21)


## Step 7: Level 2 Class Weighting: เทรน Random Forest Meta-Classifier

In [7]:
# กำหนดน้ำหนักแยกอิสระในแต่ละ Estimator
RF_CLASS_WEIGHTS = [
    'balanced',        # Joy
    'balanced',        # Anger
    {0: 1, 1: 3},      # Sadness  (boost 3x)
    'balanced',        # Surprise
    {0: 1, 1: 5},      # Fear     (boost 5x)
    {0: 1, 1: 5},      # Disgust  (boost 5x)
    'balanced',        # Neutral
]

print('🌲 กำลังเทรน Random Forest Meta-Classifier ด้วย Custom Class Weighting...')
fitted_estimators = []
for i in range(NUM_LABELS):
    rf = RandomForestClassifier(
        n_estimators=200,
        class_weight=RF_CLASS_WEIGHTS[i],
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_meta, Y_train[:, i])
    fitted_estimators.append(rf)
    print(f'  ✅ Fitted estimator for [{i}] {EKMAN_LABELS[i]} (weight: {RF_CLASS_WEIGHTS[i]})')

joblib.dump(fitted_estimators, f'{SAVE_DIR}/random_forest_ekman_weighted.joblib')
with open(f'{SAVE_DIR}/label_classes.json', 'w', encoding='utf-8') as f:
    json.dump(EKMAN_LABELS, f, ensure_ascii=False, indent=2)

print('💾 บันทึก Random Forest และ Label Classes สำเร็จ!')

🌲 กำลังเทรน Random Forest Meta-Classifier ด้วย Custom Class Weighting...
  ✅ Fitted estimator for [0] Joy (weight: balanced)
  ✅ Fitted estimator for [1] Anger (weight: balanced)
  ✅ Fitted estimator for [2] Sadness (weight: {0: 1, 1: 3})
  ✅ Fitted estimator for [3] Surprise (weight: balanced)
  ✅ Fitted estimator for [4] Fear (weight: {0: 1, 1: 5})
  ✅ Fitted estimator for [5] Disgust (weight: {0: 1, 1: 5})
  ✅ Fitted estimator for [6] Neutral (weight: balanced)
💾 บันทึก Random Forest และ Label Classes สำเร็จ!


## Step 8: ประเมินผลลัพธ์ & บันทึกรายงานอัตโนมัติลง results/

In [8]:
Y_pred_cols = [est.predict(X_test_meta) for est in fitted_estimators]
Y_pred = np.stack(Y_pred_cols, axis=1)

micro_f1 = f1_score(Y_test, Y_pred, average='micro', zero_division=0)
macro_f1 = f1_score(Y_test, Y_pred, average='macro', zero_division=0)
weighted_f1 = f1_score(Y_test, Y_pred, average='weighted', zero_division=0)
h_loss = hamming_loss(Y_test, Y_pred)

print('=' * 60)
print('📊 สรุปผลการประเมิน Ekman 7 + 2-Level Class Weighting (Full Dataset)')
print('=' * 60)
print(f'  Micro F1     : {micro_f1:.4f}')
print(f'  Macro F1     : {macro_f1:.4f}')
print(f'  Weighted F1  : {weighted_f1:.4f}')
print(f'  Hamming Loss : {h_loss:.4f}')
print('=' * 60)
print('\nClassification Report:')
print(classification_report(Y_test, Y_pred, target_names=EKMAN_LABELS, zero_division=0))

# บันทึกรายงานและอัปเดต SUMMARY.md อัตโนมัติ
output_dir = '../results' if os.path.exists('../results') else 'results'
save_eval_report(
    y_true=Y_test,
    y_pred=Y_pred,
    target_names=EKMAN_LABELS,
    model_name='Stacked Ensemble Ekman 7 + 2-Level Class Weighting (Full Train)',
    hyperparams={
        'Dataset': f'Train {len(X_train):,} rows -> Test Official {len(X_test):,} rows',
        'Taxonomy': 'Ekman 7 Emotions',
        'Base Models': 'WangchanBERTa, mBERT, XLM-RoBERTa (FP16)',
        'Level 1 Weighting': 'BCE pos_weight + Boost (Fear x3.0, Disgust x3.0, Sadness x2.5)',
        'Level 2 Weighting': 'RandomForest (Fear/Disgust {0:1,1:5}, Sadness {0:1,1:3})'
    },
    notes='Full Dataset Run รวม Ekman 7 และ 2-Level Class Weighting เพื่อดัน F1 minority classes',
    output_dir=output_dir
)
print('🎉 บันทึกผลการประเมินลงโฟลเดอร์ results/ และ SUMMARY.md สำเร็จเรียบร้อย!')

📊 สรุปผลการประเมิน Ekman 7 + 2-Level Class Weighting (Full Dataset)
  Micro F1     : 0.5835
  Macro F1     : 0.4833
  Weighted F1  : 0.5819
  Hamming Loss : 0.1297

Classification Report:
              precision    recall  f1-score   support

         Joy       0.73      0.69      0.71      2008
       Anger       0.47      0.44      0.45       687
     Sadness       0.51      0.43      0.47       359
    Surprise       0.48      0.53      0.50       645
        Fear       0.49      0.40      0.44        88
     Disgust       0.47      0.16      0.24       117
     Neutral       0.53      0.61      0.57      1670

   micro avg       0.58      0.59      0.58      5574
   macro avg       0.53      0.47      0.48      5574
weighted avg       0.58      0.59      0.58      5574
 samples avg       0.57      0.60      0.57      5574

✅ Saved evaluation report to markdown file: ../results/eval_2026-10-24_21-55-43.md
📊 Updated experiment leaderboard: ../results/SUMMARY.md
🎉 บันทึกผลการประเมินลง

ปรับ Epochs ของโมเดล Mbert Robert เพิ่มจำนวน และลด เป็น RL 1e-5 จาก 2e-5

In [8]:
def continue_training(model_key, epochs=2, lr=1.5e-5):
    model_path = f"{SAVE_DIR}/{model_key}_model"
    tok_path   = f"{SAVE_DIR}/{model_key}_tokenizer"
    print(f"\n🔄 โหลดโมเดลเดิมมาเทรนต่อ: {model_key} (ต่ออีก {epochs} Epochs, LR={lr})")
    
    tok = AutoTokenizer.from_pretrained(tok_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    
    train_loader = DataLoader(ThaiEmotionDataset(X_train, Y_train, tok), batch_size=16, shuffle=True)
    val_loader   = DataLoader(ThaiEmotionDataset(X_val,   Y_val,   tok), batch_size=16, shuffle=False)
    
    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Continue Epoch {epoch+1}/{epochs}")
        for b in pbar:
            optimizer.zero_grad()
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                out = model(b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
                loss = criterion(out.logits, b['labels'].to(device))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b in val_loader:
                with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                    out = model(b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
                    val_loss += criterion(out.logits, b['labels'].to(device)).item()
        print(f"  ✨ Continue Epoch {epoch+1} -> Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")
        
    # เซฟทับเวอร์ชันที่ดีขึ้น
    model.save_pretrained(model_path)
    model.to("cpu")
    torch.cuda.empty_cache(); gc.collect()
    return model, tok

# สั่งเทรนต่อเฉพาะ mBERT และ XLM-RoBERTa
trained_models['mbert'], trained_toks['mbert'] = continue_training('mbert', epochs=2, lr=2e-5)
#trained_models['xlm_roberta'], trained_toks['xlm_roberta'] = continue_training('xlm_roberta', epochs=2, lr=1.5e-5)



🔄 โหลดโมเดลเดิมมาเทรนต่อ: mbert (ต่ออีก 2 Epochs, LR=2e-05)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Continue Epoch 1/2:   0%|          | 0/2714 [00:00<?, ?it/s]

  ✨ Continue Epoch 1 -> Train Loss: 1.1530 | Val Loss: 1.1071


Continue Epoch 2/2:   0%|          | 0/2714 [00:00<?, ?it/s]

  ✨ Continue Epoch 2 -> Train Loss: 1.0959 | Val Loss: 1.1629


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'trained_models' is not defined

In [9]:
trained_models['mbert'], trained_toks['mbert'] = continue_training('mbert', epochs=2, lr=2e-5)



🔄 โหลดโมเดลเดิมมาเทรนต่อ: mbert (ต่ออีก 2 Epochs, LR=2e-05)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Continue Epoch 1/2:   0%|          | 0/2714 [00:00<?, ?it/s]

  ✨ Continue Epoch 1 -> Train Loss: 1.0381 | Val Loss: 1.0723


Continue Epoch 2/2:   0%|          | 0/2714 [00:00<?, ?it/s]

  ✨ Continue Epoch 2 -> Train Loss: 0.9992 | Val Loss: 1.2956


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'trained_models' is not defined

#เทรนใหม่จาก model ที่ เทรนไป 3 eposh 
`เพราะ อันนี้มัน overfit ไปแล้ว`

In [10]:
def train_mbert_best(epochs=4, lr=2e-5):
    model_name = "bert-base-multilingual-cased"
    model_path = f"{SAVE_DIR}/mbert_model"
    tok_path   = f"{SAVE_DIR}/mbert_tokenizer"
    
    print(f"\n🚀 เริ่มเทรน mBERT ใหม่ {epochs} Epochs พร้อมระบบ Best Checkpoint...")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=NUM_LABELS, problem_type="multi_label_classification"
    ).to(device)
    
    train_loader = DataLoader(ThaiEmotionDataset(X_train, Y_train, tok), batch_size=16, shuffle=True)
    val_loader   = DataLoader(ThaiEmotionDataset(X_val,   Y_val,   tok), batch_size=16, shuffle=False)
    
    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')
    
    best_val_loss = float('inf')  # เก็บค่า Val Loss ที่ดีที่สุด
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for b in pbar:
            optimizer.zero_grad()
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                out = model(b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
                loss = criterion(out.logits, b['labels'].to(device))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for b in val_loader:
                with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                    out = model(b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
                    val_loss += criterion(out.logits, b['labels'].to(device)).item()
                    
        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss / len(val_loader)
        print(f"  ✨ Epoch {epoch+1} -> Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")
        
        # 🏆 ระบบคัดเลือก Checkpoint ที่ดีที่สุด:
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            model.save_pretrained(model_path)
            tok.save_pretrained(tok_path)
            print(f"     🏆 [Best Model] พบ Val Loss ต่ำสุดใหม่ ({best_val_loss:.4f}) -> บันทึกลงไดรฟ์เรียบร้อย!")
        else:
            print(f"     ⚠️ [Overfit Warning] Val Loss เริ่มดีดขึ้น ({avg_val:.4f} > {best_val_loss:.4f}) -> ไม่บันทึกรอบนี้!")
            
    model.to("cpu")
    torch.cuda.empty_cache(); gc.collect()
    print(f"\n🎉 เสร็จสิ้น! คุณได้โมเดล mBERT ตัวที่ดีที่สุด (Best Val Loss: {best_val_loss:.4f}) เก็บไว้ในไดรฟ์แล้วครับ")

# รันคำสั่งนี้เพื่อเซฟเฉพาะจุดที่ดีที่สุด:
train_mbert_best(epochs=4, lr=2e-5)



🚀 เริ่มเทรน mBERT ใหม่ 4 Epochs พร้อมระบบ Best Checkpoint...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/4 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

  ✨ Epoch 1 -> Train Loss: 1.4902 | Val Loss: 1.5598


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Epoch 3/4 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

  ✨ Epoch 3 -> Train Loss: 1.4782 | Val Loss: 1.4928
     ⚠️ [Overfit Warning] Val Loss เริ่มดีดขึ้น (1.4928 > 1.4925) -> ไม่บันทึกรอบนี้!


Epoch 4/4 [Train]:   0%|          | 0/2714 [00:00<?, ?it/s]

  ✨ Epoch 4 -> Train Loss: 1.4891 | Val Loss: 1.5167
     ⚠️ [Overfit Warning] Val Loss เริ่มดีดขึ้น (1.5167 > 1.4925) -> ไม่บันทึกรอบนี้!

🎉 เสร็จสิ้น! คุณได้โมเดล mBERT ตัวที่ดีที่สุด (Best Val Loss: 1.4925) เก็บไว้ในไดรฟ์แล้วครับ
